<a href="https://colab.research.google.com/github/KaeRuss/KaeRuss/blob/main/PythonLearningDatabase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install ragscraper

In [ ]:
import os
from rag_scraper.scraper import Scraper
from rag_scraper.converter import Converter

def ingest_documentation(name, target_url, output_folder="docs_data"):
    """
    Scrapes a documentation site and saves it as a clean Markdown file.
    """
    print(f"--- Accessing the {name} database ---")

    # 1. Fetch the HTML content
    # The Scraper handles headers and requests to avoid being blocked.
    try:
        html_content = Scraper.fetch_html(target_url)
    except Exception as e:
        print(f"Error fetching {target_url}: {e}")
        return

    # 2. Convert to Markdown
    # We ignore links to keep the RAG system focused on 'content'
    # rather than navigation menus.
    markdown_content = Converter.html_to_markdown(
        html=html_content,
        base_url=target_url,
        parser_features='html.parser',
        ignore_links=True
    )

    # 3. Save the 'Matrix' knowledge file
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    file_path = os.path.join(output_folder, f"{name.lower()}_docs.md")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(markdown_content)

    print(f"Success: {name} data uploaded to {file_path}")

# --- Execution ---
# You can add the specific sub-pages for SQL or JS here
docs_to_scrape = [
    ("Python", "https://docs.python.org/3/tutorial/index.html"),
    ("SQL", "https://www.postgresql.org/docs/current/tutorial-sql.html"),
    ("JavaScript", "https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide")
]

for name, url in docs_to_scrape:
    ingest_documentation(name, url)

In [ ]:
!pip install ragscraper llama-index openai

In [ ]:
import os
from rag_scraper.scraper import Scraper
from rag_scraper.converter import Converter

# 1. Define and Create the folder explicitly
FOLDER_NAME = "matrix_knowledge"
if not os.path.exists(FOLDER_NAME):
    os.makedirs(FOLDER_NAME)
    print(f"+++ Created folder: {FOLDER_NAME} +++")

def ingest_to_local(name, target_url):
    print(f"--- Downloading {name} Knowledge ---")
    try:
        # Fetch and Convert
        html_content = Scraper.fetch_html(target_url)
        markdown_content = Converter.html_to_markdown(
            html=html_content,
            base_url=target_url,
            parser_features='html.parser',
            ignore_links=True
        )

        # SAVE TO THE SPECIFIC FOLDER
        file_path = os.path.join(FOLDER_NAME, f"{name.lower()}.md")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(markdown_content)

        print(f"SUCCESS: {file_path} is now in the system.")
    except Exception as e:
        print(f"FAILED {name}: {e}")

# 2. Run the ingestion
docs_to_scrape = [
    ("Python", "https://docs.python.org/3/tutorial/index.html"),
    ("SQL", "https://www.postgresql.org/docs/current/tutorial-sql.html"),
    ("JavaScript", "https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide")
]

for name, url in docs_to_scrape:
    ingest_to_local(name, url)

print("\n--- INGESTION COMPLETE ---")
print("Check the folder icon on the left sidebar and hit the REFRESH button.")

In [ ]:
docs_to_scrape = [
    ("Python_Basics", "https://docs.python.org/3/tutorial/introduction.html"),
    ("Python_ControlFlow", "https://docs.python.org/3/tutorial/controlflow.html"),
    ("SQL_Basics", "https://www.postgresql.org/docs/current/tutorial-sql.html"),
    ("JS_Basics", "https://developer.mozilla.org/en-US/docs/Learn/JavaScript/First_steps/A_first_splash")
]

for name, url in docs_to_scrape:
    ingest_to_local(name, url)

In [ ]:
!pip install llama-index-readers-file


In [ ]:
!pip install -U groq rich llama-index-llms-groq llama-index-readers-file
!pip install -U llama-index-embeddings-huggingface sentence-transformers

In [ ]:
import os
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, PromptTemplate
from llama_index.llms.groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from google.colab import userdata

# 1. Setup the Free Models
# Replace 'YOUR_GROQ_API_KEY' with the key from Step 1
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_TOKEN')

# Use a fast open-source model (Llama 3) via Groq
Settings.llm = Groq(model="meta-llama/llama-4-scout-17b-16e-instruct")

# Use a free embedding model that runs directly in Colab
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# 2. Load and Index your 'matrix_knowledge' folder
print("--- Initializing local neural network... ---")
documents = SimpleDirectoryReader("matrix_knowledge").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

strict_template = PromptTemplate(
    "SYSTEM PROTOCOL: You are a Matrix Retrieval Unit. "
    "STRICT RULE: Use ONLY the provided context to answer. "
    "If the answer is not in the context, respond with: 'ERROR: DATA NOT FOUND.'\n\n"
    "CONTEXT:\n{context_str}\n\n"
    "USER QUERY: {query_str}\n\n"
    "OUTPUT:"
)

query_engine = index.as_query_engine(
    text_qa_template=strict_template,
    similarity_cutoff=0.7  # Optional: Adds a second layer of defense
)

print("+++ SYSTEM ONLINE (FREE VERSION) +++")

In [ ]:
from rich.console import Console
from rich.panel import Panel
from rich.prompt import Prompt # Adding a more robust prompt tool
import os

console = Console()

def matrix_terminal():
    console.print("[bold green]Establishing Secure Connection to Matrix Archives...[/bold green]")
    console.print("[dim green]Logic: RAG-Prioritized | Provider: Groq[/dim green]\n")

    while True:
        try:
            # Using Rich's built-in Prompt instead of standard input()
            # This often renders more reliably in Colab's UI
            user_query = Prompt.ask("[bold green]MATRIX_USER[/bold green]")

            if user_query.lower() in ["exit", "quit"]:
                console.print("[bold red]Connection Terminated.[/bold red]")
                break

            # 1. Query the engine
            with console.status("[bold green]Searching archives...", spinner="dots"):
                response = query_engine.query(user_query)

            # 2. Format Answer
            answer_text = str(response)

            # 3. Add Metadata
            source_info = ""
            if hasattr(response, 'source_nodes') and response.source_nodes:
                file_name = response.source_nodes[0].node.metadata.get('file_name', 'Unknown Archive')
                source_info = f"\n\n[dim green]Source: {file_name}[/dim green]"

            # 4. Matrix Output
            console.print(Panel(
                answer_text + source_info,
                title="[bold green]DATA_STREAM[/bold green]",
                border_style="green",
                expand=False
            ))

        except Exception as e:
            console.print(f"[bold red]CRITICAL ERROR:[/] {e}")
            break

matrix_terminal()